# Problema 2 — La Trampa Trabajo-Agua-Salud
### ODS 6 · ODS 8 · ODS 12 · Narrativa Visual con Plotly · HackODS 2026

> *"México crece. Su consumo material se multiplicó por 18 en 20 años.  
> Al mismo tiempo, su acceso al agua segura retrocedió, sus cuerpos de agua se encogen,  
> y sus aguas residuales volvieron a quedar sin tratar.  
> No son hechos separados. Son el mismo hecho."*

---

**El problema:** Los datos del INEGI revelan correlaciones fuertemente negativas entre el crecimiento económico (ODS 8), el acceso al agua (ODS 6) y la salud hídrica de la población (ODS 3). Conforme avanzan los indicadores de producción y consumo, los indicadores de agua y salud retroceden.  
Esto es la definición de un modelo de desarrollo **insostenible**.

**Fuente:** INEGI — 333 indicadores ODS de México (1970–2026)  
**Método:** Correlaciones Spearman, |r| ≥ 0.7, p < 0.05

In [1]:
import pandas as pd
import numpy as np
import ast
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Paleta de colores ─────────────────────────────────────────────────────────
C_AGUA    = '#1D6FA4'   # azul agua ODS 6
C_TRABAJO = '#E5891A'   # naranja trabajo ODS 8
C_SALUD   = '#4C9F38'   # verde salud ODS 3
C_CONSUMO = '#8B1A1A'   # rojo oscuro consumo/daño ODS 12
C_ALERTA  = '#E63946'   # rojo alarma
C_MEJORA  = '#2DC653'   # verde mejora
C_NEUTRO  = '#6C757D'   # gris
C_FONDO   = '#F8F9FA'   # fondo
C_LINEA   = '#1D3557'   # texto

# ── Cargar datos ──────────────────────────────────────────────────────────────
df   = pd.read_csv('../data/consolidated/indicadores_largo.csv')
corr = pd.read_csv('../data/consolidated/correlaciones_altas.csv')
pares = pd.read_csv('../data/consolidated/pares_ods_correlaciones.csv')

def get_serie(ind_id):
    return df[df['indicator_id'] == ind_id].sort_values('periodo').copy()

print('Datos cargados:', len(df), 'observaciones')

# Resumen del trilema en datos
pares['pair_tuple'] = pares['ods_pair'].apply(ast.literal_eval)
pares['ods_1'] = pares['pair_tuple'].apply(lambda x: x[0])
pares['ods_2'] = pares['pair_tuple'].apply(lambda x: x[1])

def get_pair(a, b):
    r = pares[((pares['ods_1']==a)&(pares['ods_2']==b)) |
              ((pares['ods_1']==b)&(pares['ods_2']==a))]
    return r.iloc[0] if len(r) > 0 else None

p83  = get_pair(8,3)
p86  = get_pair(8,6)
p123 = get_pair(12,3)
p126 = get_pair(12,6)
print(f'ODS 8-3: {int(p83["n_correlaciones"])} pares, corr media {p83["corr_promedio"]:.3f}')
print(f'ODS 8-6: {int(p86["n_correlaciones"])} pares, corr media {p86["corr_promedio"]:.3f}')
print(f'ODS 12-3: {int(p123["n_correlaciones"])} pares, corr media {p123["corr_promedio"]:.3f}')
print(f'ODS 12-6: {int(p126["n_correlaciones"])} pares, corr media {p126["corr_promedio"]:.3f}')

Datos cargados: 3611 observaciones
ODS 8-3: 177 pares, corr media -0.068
ODS 8-6: 88 pares, corr media -0.075
ODS 12-3: 48 pares, corr media -0.273
ODS 12-6: 29 pares, corr media -0.043


---
## Resumen Ejecutivo — Las 4 cifras del Trilema

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 1 — KPIs del Trilema (4 gauges)
# ══════════════════════════════════════════════════════════════════════════════

fig_kpi = make_subplots(
    rows=1, cols=4,
    subplot_titles=[
        'Consumo material per capita',
        'Acceso agua potable segura',
        'Estres hidrico nacional',
        'Aguas residuales tratadas'
    ],
    specs=[[{'type':'indicator'}]*4],
    horizontal_spacing=0.06
)

# KPI 1: Consumo material per cápita (explosion)
fig_kpi.add_trace(go.Indicator(
    mode='number+delta+gauge',
    value=214.0,
    delta={'reference': 11.5, 'valueformat':'.0f', 'suffix':' ton',
           'increasing':{'color': C_ALERTA}},
    number={'suffix':' ton', 'valueformat':'.0f',
            'font':{'size':36, 'color': C_ALERTA}},
    gauge={
        'axis':{'range':[0,250]},
        'bar':{'color': C_ALERTA,'thickness':0.35},
        'bgcolor':'#F0F0F0',
        'steps':[
            {'range':[0,50],  'color':'rgba(44,198,83,0.15)'},
            {'range':[50,120],'color':'rgba(244,162,97,0.15)'},
            {'range':[120,250],'color':'rgba(230,57,70,0.15)'}
        ],
        'threshold':{'line':{'color':'#888','width':2},'value':11.5}
    },
    title={'text':'por persona en 2024 | +1,763% vs 2003'}
), row=1, col=1)

# KPI 2: Acceso agua potable segura (retroceso)
fig_kpi.add_trace(go.Indicator(
    mode='number+delta+gauge',
    value=61.0,
    delta={'reference': 66.9, 'valueformat':'.1f', 'suffix':' pp',
           'decreasing':{'color': C_ALERTA}},
    number={'suffix':'%', 'valueformat':'.0f',
            'font':{'size':40, 'color': C_ALERTA}},
    gauge={
        'axis':{'range':[50,100], 'ticksuffix':'%'},
        'bar':{'color': C_AGUA,'thickness':0.35},
        'bgcolor':'#F0F0F0',
        'steps':[
            {'range':[50,65], 'color':'rgba(230,57,70,0.15)'},
            {'range':[65,80], 'color':'rgba(244,162,97,0.15)'},
            {'range':[80,100],'color':'rgba(44,198,83,0.15)'}
        ],
        'threshold':{'line':{'color': C_MEJORA,'width':3},'value':95}
    },
    title={'text':'con agua segura en 2022 | meta: 95%'}
), row=1, col=2)

# KPI 3: Estrés hídrico
fig_kpi.add_trace(go.Indicator(
    mode='number+delta+gauge',
    value=47.2,
    delta={'reference': 46.4, 'valueformat':'.1f',
           'increasing':{'color': C_ALERTA}},
    number={'suffix':'%', 'valueformat':'.1f',
            'font':{'size':40, 'color': C_ALERTA}},
    gauge={
        'axis':{'range':[0,100], 'ticksuffix':'%'},
        'bar':{'color': C_ALERTA,'thickness':0.35},
        'bgcolor':'#F0F0F0',
        'steps':[
            {'range':[0,25],  'color':'rgba(44,198,83,0.15)'},
            {'range':[25,40], 'color':'rgba(244,162,97,0.15)'},
            {'range':[40,100],'color':'rgba(230,57,70,0.15)'}
        ],
        'threshold':{'line':{'color': C_ALERTA,'width':3},'value':40}
    },
    title={'text':'extraccion vs disponibilidad | umbral critico: 40%'}
), row=1, col=3)

# KPI 4: Aguas residuales tratadas
fig_kpi.add_trace(go.Indicator(
    mode='number+delta+gauge',
    value=40.1,
    delta={'reference': 49.6, 'valueformat':'.1f', 'suffix':' pp',
           'decreasing':{'color': C_ALERTA}},
    number={'suffix':'%', 'valueformat':'.1f',
            'font':{'size':40, 'color': C_ALERTA}},
    gauge={
        'axis':{'range':[0,100], 'ticksuffix':'%'},
        'bar':{'color': C_AGUA,'thickness':0.35},
        'bgcolor':'#F0F0F0',
        'steps':[
            {'range':[0,40],  'color':'rgba(230,57,70,0.15)'},
            {'range':[40,70], 'color':'rgba(244,162,97,0.15)'},
            {'range':[70,100],'color':'rgba(44,198,83,0.15)'}
        ],
        'threshold':{'line':{'color': C_MEJORA,'width':3},'value':70}
    },
    title={'text':'tratadas adecuadamente en 2023 | retroceso desde 2018'}
), row=1, col=4)

fig_kpi.update_layout(
    title=dict(
        text='<b>La Trampa Trabajo-Agua-Salud — Las 4 Cifras Clave</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'El crecimiento del consumo convive con el deterioro del agua</span>',
        x=0.5, xanchor='center'
    ),
    height=390, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA)
)
fig_kpi.show()

---
## Parte I — El Consumo que Explota

### ODS 12: Producción y Consumo Responsables

El consumo material interno per cápita de México pasó de **11.5 toneladas** por persona en 2003 a **214 toneladas** en 2024.  
Un incremento de **+1,763%** en dos décadas.

¿Por qué importa esto para el agua? Porque extraer, procesar, transportar y desechar materiales consume agua y la contamina.  
Los datos confirman: el consumo material tiene correlaciones **perfectamente negativas** (r = -1.0) con el acceso a agua potable segura.

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 2 — Consumo material per cápita (explosión 2003–2024)
# ══════════════════════════════════════════════════════════════════════════════

cmat = get_serie('12.2.2.B')   # consumo material per cápita (toneladas)
pib  = get_serie('8.1.1')      # crecimiento PIB per cápita

fig_cmat = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Consumo Material Interno per Capita (toneladas/persona)',
        'Tasa de Crecimiento del PIB Real per Capita (%)'
    ],
    horizontal_spacing=0.12
)

# ── Consumo material ──────────────────────────────────────────────────────────
# Separar en dos fases para resaltar la aceleración
cmat_fase1 = cmat[cmat['periodo'] <= 2014]
cmat_fase2 = cmat[cmat['periodo'] >= 2014]

fig_cmat.add_trace(go.Scatter(
    x=cmat_fase1['periodo'], y=cmat_fase1['valor'],
    mode='lines+markers', name='Crecimiento moderado (2003-2014)',
    line=dict(color=C_TRABAJO, width=2.5),
    marker=dict(size=7, color=C_TRABAJO, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(229,137,26,0.08)',
    hovertemplate='<b>%{x}</b><br>Consumo: <b>%{y:.1f}</b> ton/persona<extra></extra>'
), row=1, col=1)

fig_cmat.add_trace(go.Scatter(
    x=cmat_fase2['periodo'], y=cmat_fase2['valor'],
    mode='lines+markers', name='Aceleracion exponencial (2014-2024)',
    line=dict(color=C_ALERTA, width=3),
    marker=dict(size=8, color=C_ALERTA, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(230,57,70,0.10)',
    hovertemplate='<b>%{x}</b><br>Consumo: <b>%{y:.1f}</b> ton/persona<extra></extra>'
), row=1, col=1)

# Anotaciones clave
fig_cmat.add_annotation(
    x=2003, y=11.5,
    text='<b>11.5 ton</b><br>2003',
    showarrow=True, arrowhead=2, arrowcolor=C_NEUTRO,
    ax=60, ay=-35, font=dict(size=10, color=C_NEUTRO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_NEUTRO, borderwidth=1,
    row=1, col=1
)
fig_cmat.add_annotation(
    x=2024, y=214.0,
    text='<b>214 ton</b><br>2024 (+1,763%)',
    showarrow=True, arrowhead=2, arrowcolor=C_ALERTA,
    ax=-70, ay=-40, font=dict(size=11, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=1, col=1
)
fig_cmat.add_annotation(
    x=2016, y=80,
    text='<b>x18</b><br>en 20 años',
    showarrow=False, font=dict(size=20, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.7)',
    row=1, col=1
)

# ── PIB per cápita ────────────────────────────────────────────────────────────
# Barras coloreadas por positivo/negativo
colors_pib = [C_MEJORA if v > 0 else C_ALERTA for v in pib['valor']]
fig_cmat.add_trace(go.Bar(
    x=pib['periodo'], y=pib['valor'],
    name='Crecimiento PIB per capita',
    marker_color=colors_pib,
    marker_line_color='white', marker_line_width=0.5,
    hovertemplate='<b>%{x}</b><br>Variacion PIB: <b>%{y:.2f}%</b><extra></extra>',
    showlegend=False
), row=1, col=2)

fig_cmat.add_hline(y=0, line_width=1.5, line_color='#ADB5BD', row=1, col=2)

# Anotar las crisis
for año, val, etq in [(2009, -7.75, '2009\nCrisis global'), (2020, -7.75, '2020\nCOVID')]:
    fig_cmat.add_annotation(
        x=año, y=val - 0.5,
        text='<b>' + etq + '</b>',
        showarrow=False, font=dict(size=9, color=C_ALERTA),
        bgcolor='rgba(255,255,255,0.8)', bordercolor=C_ALERTA, borderwidth=1,
        row=1, col=2
    )

fig_cmat.add_annotation(
    x=2012, y=3.5,
    text='El PIB crece modestamente (+1-3%/año)<br>pero el consumo material<br>crece de forma exponencial',
    showarrow=False, font=dict(size=10, color=C_LINEA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor='#ADB5BD', borderwidth=1,
    align='left', row=1, col=2
)

fig_cmat.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig_cmat.update_yaxes(showgrid=True, gridcolor='#E9ECEF')
fig_cmat.update_yaxes(title_text='Toneladas por persona', row=1, col=1)
fig_cmat.update_yaxes(title_text='Variacion anual (%)', row=1, col=2)

fig_cmat.update_layout(
    title=dict(
        text='<b>ODS 12 — El Consumo que Explota (2003–2024)</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'El consumo material per capita crece exponencialmente mientras el PIB crece a ritmo modesto</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(248,249,250,0.9)', borderwidth=1),
    margin=dict(t=100, b=60)
)
fig_cmat.show()

---
## Parte II — El Agua que Retrocede

### ODS 6: Agua Limpia y Saneamiento

Mientras el consumo material se dispara, el agua enfrenta una presión creciente en tres frentes:

1. **El acceso a agua potable segura disminuye** — solo el 61% de la población en 2022, bajando desde 66.9% en 2014
2. **El estrés hídrico supera el umbral crítico** — México extrae el 47% de su agua disponible (el límite de alerta es 40%)
3. **Las aguas residuales tratadas retrocedieron** — del 50% en 2018 al 40% en 2023: más contaminación regresa a los cuerpos de agua

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 3 — Triple panel: Agua potable segura + Estrés hídrico + Aguas residuales
# ══════════════════════════════════════════════════════════════════════════════

agua_seg = get_serie('6.1.1.A')   # acceso agua potable segura
estres   = get_serie('6.4.2')     # estres hidrico (%)
presion  = get_serie('6N.2.1')    # presion hidrica zonas C/N
residual = get_serie('6.3.1')     # aguas residuales tratadas

fig_agua = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Acceso a Agua Potable Segura (%)',
        'Estres Hidrico — extraccion vs disponibilidad',
        'Aguas Residuales Tratadas Adecuadamente (%)'
    ],
    horizontal_spacing=0.1
)

# ── Panel 1: Agua potable segura ──────────────────────────────────────────────
fig_agua.add_trace(go.Scatter(
    x=agua_seg['periodo'], y=agua_seg['valor'],
    mode='lines+markers',
    line=dict(color=C_ALERTA, width=3),
    marker=dict(size=10, color=C_ALERTA, symbol='diamond',
                line=dict(width=2, color='white')),
    fill='tozeroy', fillcolor='rgba(29,111,164,0.08)',
    name='Agua potable segura',
    hovertemplate='<b>%{x}</b><br>Acceso: <b>%{y:.1f}%</b><extra></extra>',
    showlegend=False
), row=1, col=1)

fig_agua.add_hrect(
    y0=95, y1=105, fillcolor='rgba(44,198,83,0.1)', line_width=0,
    annotation_text='Meta ODS: 95%', annotation_position='top right',
    annotation_font=dict(size=9, color=C_MEJORA), row=1, col=1
)
fig_agua.add_annotation(
    x=2014, y=66.9, text='<b>66.9%</b><br>2014',
    showarrow=True, arrowhead=2, arrowcolor=C_NEUTRO,
    ax=45, ay=-30, font=dict(size=10, color=C_NEUTRO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_NEUTRO, borderwidth=1,
    row=1, col=1
)
fig_agua.add_annotation(
    x=2022, y=61.0, text='<b>61.0%</b><br>2022',
    showarrow=True, arrowhead=2, arrowcolor=C_ALERTA,
    ax=-45, ay=35, font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1,
    row=1, col=1
)

# ── Panel 2: Estrés hídrico (doble serie: nacional + zonas C/N) ───────────────
fig_agua.add_trace(go.Scatter(
    x=estres['periodo'], y=estres['valor'],
    mode='lines+markers', name='Estres hidrico nacional',
    line=dict(color=C_AGUA, width=2.5),
    marker=dict(size=8, color=C_AGUA, line=dict(width=1.5, color='white')),
    hovertemplate='<b>%{x}</b><br>Estres nacional: <b>%{y:.1f}%</b><extra></extra>',
    showlegend=False
), row=1, col=2)

fig_agua.add_trace(go.Scatter(
    x=presion['periodo'], y=presion['valor'],
    mode='lines+markers', name='Zonas Centro y Norte',
    line=dict(color=C_ALERTA, width=2.5, dash='dot'),
    marker=dict(size=7, color=C_ALERTA, line=dict(width=1.5, color='white')),
    hovertemplate='<b>%{x}</b><br>Presion C/N: <b>%{y:.1f}%</b><extra></extra>',
    showlegend=False
), row=1, col=2)

# Umbral crítico 40%
fig_agua.add_hrect(
    y0=40, y1=100, fillcolor='rgba(230,57,70,0.06)', line_width=0,
    annotation_text='Zona critica (>40%)',
    annotation_position='top left',
    annotation_font=dict(size=9, color=C_ALERTA), row=1, col=2
)
fig_agua.add_hline(
    y=40, line_dash='dash', line_color=C_ALERTA, line_width=2,
    row=1, col=2
)
fig_agua.add_annotation(
    x=2017, y=38, text='Umbral de alerta: 40%',
    showarrow=False, font=dict(size=9, color=C_ALERTA),
    row=1, col=2
)
fig_agua.add_annotation(
    x=2019, y=53,
    text='Zonas C/N<br><b>55-56%</b>\n(ultracritico)',
    showarrow=True, arrowhead=2, arrowcolor=C_ALERTA,
    ax=50, ay=-30, font=dict(size=9, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1,
    row=1, col=2
)

# ── Panel 3: Aguas residuales tratadas ───────────────────────────────────────
res_sube = residual[residual['periodo'] <= 2018]
res_baja = residual[residual['periodo'] >= 2018]

fig_agua.add_trace(go.Scatter(
    x=res_sube['periodo'], y=res_sube['valor'],
    mode='lines+markers', name='Tendencia positiva',
    line=dict(color=C_MEJORA, width=2.5),
    marker=dict(size=6, color=C_MEJORA, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(44,198,83,0.06)',
    hovertemplate='<b>%{x}</b><br>Tratadas: <b>%{y:.1f}%</b><extra></extra>',
    showlegend=False
), row=1, col=3)

fig_agua.add_trace(go.Scatter(
    x=res_baja['periodo'], y=res_baja['valor'],
    mode='lines+markers', name='Retroceso post-2018',
    line=dict(color=C_ALERTA, width=3),
    marker=dict(size=7, color=C_ALERTA, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(230,57,70,0.06)',
    hovertemplate='<b>%{x}</b><br>Tratadas: <b>%{y:.1f}%</b><extra></extra>',
    showlegend=False
), row=1, col=3)

fig_agua.add_annotation(
    x=2018, y=49.6,
    text='<b>Pico 49.6%</b><br>2018',
    showarrow=True, arrowhead=2, arrowcolor=C_MEJORA,
    ax=45, ay=-35, font=dict(size=10, color=C_MEJORA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_MEJORA, borderwidth=1,
    row=1, col=3
)
fig_agua.add_annotation(
    x=2023, y=40.1,
    text='<b>40.1%</b><br>2023<br>(-9.5 pp)',
    showarrow=True, arrowhead=2, arrowcolor=C_ALERTA,
    ax=-50, ay=40, font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1,
    row=1, col=3
)

fig_agua.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig_agua.update_yaxes(showgrid=True, gridcolor='#E9ECEF')
fig_agua.update_yaxes(title_text='% de la poblacion', row=1, col=1)
fig_agua.update_yaxes(title_text='% de extraccion vs disponibilidad', row=1, col=2)
fig_agua.update_yaxes(title_text='% de aguas residuales', row=1, col=3)

fig_agua.update_layout(
    title=dict(
        text='<b>ODS 6 — El Agua que Retrocede en Mexico</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Tres indicadores que revelan la presion creciente sobre el recurso hidrico</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig_agua.show()

---
## Parte III — La Conexión: Consumo vs Agua vs Salud

Aquí está el corazón del análisis: las **correlaciones de Spearman** revelan que el crecimiento del consumo material no solo no mejora el acceso al agua — activamente lo deteriora, y ese deterioro tiene consecuencias en salud.

Las tres correlaciones más fuertes del sistema:
- **Consumo material ↔ Agua potable segura: r = −1.00** (5 puntos compartidos)
- **Consumo material ↔ Mortalidad por agua insalubre: r = −0.97** (21 puntos)
- **Consumo material ↔ Mortalidad neonatal: r = −0.92** (21 puntos)

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 4 — Scatter plots: Consumo material vs indicadores de agua y salud
# ══════════════════════════════════════════════════════════════════════════════

cmat  = get_serie('12.2.2.B')  # consumo material per capita
agua  = get_serie('6.1.1.A')   # agua potable segura
wash  = get_serie('3.9.2A')    # mortalidad por agua insalubre
neon  = get_serie('3.2.2')     # mortalidad neonatal
fertil = get_serie('12N.2.1')  # fertilizantes

def merge_indicadores(ind_a, ind_b):
    """Merge dos indicadores por periodo para scatter plot."""
    a = ind_a[['periodo','valor']].rename(columns={'valor':'x'})
    b = ind_b[['periodo','valor']].rename(columns={'valor':'y'})
    merged = a.merge(b, on='periodo')
    return merged

# Calcular correlación de Spearman para cada par
from scipy.stats import spearmanr

def corr_label(df_m):
    if len(df_m) < 3:
        return 'n/a'
    r, p = spearmanr(df_m['x'], df_m['y'])
    return 'r={:.2f} (n={})'.format(r, len(df_m))

fig_scatter = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Consumo material vs Acceso a agua segura',
        'Consumo material vs Mortalidad por agua insalubre',
        'Consumo material vs Mortalidad neonatal'
    ],
    horizontal_spacing=0.10
)

pares_scatter = [
    (cmat, agua,  'Agua potable segura (%)',       C_AGUA,    1),
    (cmat, wash,  'Mortalidad WASH (por 100k)',    C_SALUD,   2),
    (cmat, neon,  'Mortalidad neonatal (por 1k)',  C_ALERTA,  3),
]

for ind_a, ind_b, ylabel, color, col in pares_scatter:
    m = merge_indicadores(ind_a, ind_b)
    if len(m) == 0:
        continue
    r_val, p_val = spearmanr(m['x'], m['y']) if len(m) >= 3 else (0, 1)
    label_r = 'r = {:.2f}  (n = {})'.format(r_val, len(m))

    # Color de puntos por año
    fig_scatter.add_trace(go.Scatter(
        x=m['x'], y=m['y'],
        mode='markers+text',
        text=m['periodo'].astype(str),
        textposition='top center',
        textfont=dict(size=8, color='#6C757D'),
        marker=dict(
            size=10,
            color=m['periodo'],
            colorscale='RdYlGn_r',
            showscale=(col == 2),
            colorbar=dict(title='Año', x=0.67, thickness=12) if col == 2 else None,
            line=dict(width=1.5, color='white')
        ),
        name=ylabel,
        hovertemplate='<b>%{text}</b><br>Consumo: <b>%{x:.1f}</b> ton<br>'
                      + ylabel + ': <b>%{y:.2f}</b><extra></extra>',
        showlegend=False
    ), row=1, col=col)

    # Línea de tendencia
    if len(m) >= 3:
        coeffs = np.polyfit(m['x'], m['y'], 1)
        x_line = np.linspace(m['x'].min(), m['x'].max(), 50)
        y_line = np.polyval(coeffs, x_line)
        fig_scatter.add_trace(go.Scatter(
            x=x_line, y=y_line,
            mode='lines', line=dict(color=color, width=2, dash='dash'),
            showlegend=False, hoverinfo='skip'
        ), row=1, col=col)

    # Anotación con r
    fig_scatter.add_annotation(
        xref='x' + str(col) if col > 1 else 'x',
        yref='y' + str(col) if col > 1 else 'y',
        x=m['x'].min() + (m['x'].max() - m['x'].min()) * 0.05,
        y=m['y'].max() - (m['y'].max() - m['y'].min()) * 0.1,
        text='<b>Spearman</b><br>' + label_r,
        showarrow=False,
        font=dict(size=11, color=color),
        bgcolor='rgba(255,255,255,0.9)', bordercolor=color, borderwidth=1.5,
        row=1, col=col
    )

fig_scatter.update_xaxes(title_text='Consumo material per capita (ton)', showgrid=True, gridcolor='#E9ECEF')
fig_scatter.update_yaxes(showgrid=True, gridcolor='#E9ECEF')

fig_scatter.update_layout(
    title=dict(
        text='<b>La Conexion: Mayor Consumo = Peor Agua = Peor Salud</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Correlaciones de Spearman entre consumo material y acceso al agua / salud</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig_scatter.show()

---
## Parte IV — El Trabajo Informal: el eslabón invisible

### ODS 8: Trabajo Decente y Crecimiento Económico

Hay un mecanismo que conecta el trabajo con el agua y la salud que los datos revelan de forma indirecta:  
**el 51% de los trabajadores mexicanos están en el sector informal**.

El empleo informal significa, en la práctica:
- Sin acceso a seguridad social (salud pública)
- Sin condiciones laborales reguladas de higiene y agua
- Sin cobertura para lesiones ocupacionales
- Actividades que frecuentemente degradan el entorno hídrico local

El dato impactante: la correlación entre **personal sanitario** (ODS 3.C.1) y el **empleo informal** (ODS 8.3.1) es de **r = −0.98** — a mayor informalidad, menor densidad de personal médico.

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 5 — El trabajo informal: series y relación con salud
# ══════════════════════════════════════════════════════════════════════════════

informal = get_serie('8.3.1')    # empleo informal
desoc    = get_serie('8.5.2')    # desocupacion
lesiones = get_serie('8.8.1')    # lesiones ocupacionales
personal = get_serie('3.C.1')    # personal sanitario

fig_trab = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Empleo Informal en Mexico (%) — ODS 8.3.1',
        'Tasa de Desocupacion (%) — ODS 8.5.2',
        'Lesiones Ocupacionales (tasa) — ODS 8.8.1',
        'Personal Sanitario (por 10,000 hab.) — ODS 3.c.1'
    ],
    vertical_spacing=0.14, horizontal_spacing=0.10
)

# ── Panel 1: Informalidad ─────────────────────────────────────────────────────
fig_trab.add_trace(go.Scatter(
    x=informal['periodo'], y=informal['valor'],
    mode='lines+markers',
    line=dict(color=C_TRABAJO, width=3),
    marker=dict(size=7, color=C_TRABAJO, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(229,137,26,0.08)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Informal: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig_trab.add_hline(
    y=50, line_dash='dash', line_color=C_ALERTA, line_width=1.5,
    row=1, col=1,
    annotation_text='Linea del 50%', annotation_position='right',
    annotation_font=dict(size=9, color=C_ALERTA)
)

# Anotación: siempre por encima del 50%
fig_trab.add_annotation(
    x=2010, y=55.5,
    text='<b>Nunca por debajo del 50%</b><br>en 18 años de datos',
    showarrow=False, font=dict(size=10, color=C_TRABAJO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_TRABAJO, borderwidth=1,
    row=1, col=1
)
fig_trab.add_annotation(
    x=2024, y=50.74,
    text='<b>50.7%</b><br>2024',
    showarrow=True, arrowhead=2, arrowcolor=C_ALERTA,
    ax=-40, ay=30, font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1,
    row=1, col=1
)

# ── Panel 2: Desocupación ─────────────────────────────────────────────────────
colors_desoc = [
    C_ALERTA if v > 5 else C_TRABAJO if v > 4 else C_MEJORA
    for v in desoc['valor']
]
fig_trab.add_trace(go.Bar(
    x=desoc['periodo'], y=desoc['valor'],
    marker_color=colors_desoc,
    marker_line_color='white', marker_line_width=0.5,
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Desocupacion: <b>%{y:.2f}%</b><extra></extra>'
), row=1, col=2)

fig_trab.add_vrect(
    x0=2008.5, x1=2011.5,
    fillcolor='rgba(230,57,70,0.06)', line_width=0,
    annotation_text='Crisis global', annotation_position='top right',
    annotation_font=dict(size=9, color=C_ALERTA), row=1, col=2
)
fig_trab.add_vrect(
    x0=2019.5, x1=2021.5,
    fillcolor='rgba(230,57,70,0.06)', line_width=0,
    annotation_text='COVID', annotation_position='top left',
    annotation_font=dict(size=9, color=C_ALERTA), row=1, col=2
)
fig_trab.add_annotation(
    x=2024, y=2.7,
    text='<b>2.7%</b><br>Min historico',
    showarrow=True, arrowhead=2, arrowcolor=C_MEJORA,
    ax=-40, ay=30, font=dict(size=10, color=C_MEJORA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_MEJORA, borderwidth=1,
    row=1, col=2
)

# ── Panel 3: Lesiones ocupacionales ──────────────────────────────────────────
fig_trab.add_trace(go.Scatter(
    x=lesiones['periodo'], y=lesiones['valor'],
    mode='lines+markers',
    line=dict(color=C_NEUTRO, width=2.5),
    marker=dict(size=7, color=C_NEUTRO, line=dict(width=1.5, color='white')),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Lesiones: <b>%{y:.3f}</b> tasa<extra></extra>'
), row=2, col=1)

# Línea de tendencia
x_l = np.array(lesiones['periodo'])
y_l = np.array(lesiones['valor'])
m_l, b_l = np.polyfit(x_l, y_l, 1)
fig_trab.add_trace(go.Scatter(
    x=x_l, y=m_l*x_l + b_l,
    mode='lines', line=dict(color=C_MEJORA, width=1.5, dash='dash'),
    showlegend=False, hoverinfo='skip'
), row=2, col=1)
fig_trab.add_annotation(
    x=2012, y=2.9, text='Tendencia positiva: lesiones\nbajan (-0.07/año)',
    showarrow=False, font=dict(size=10, color=C_MEJORA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_MEJORA, borderwidth=1,
    row=2, col=1
)

# ── Panel 4: Personal sanitario ──────────────────────────────────────────────
fig_trab.add_trace(go.Scatter(
    x=personal['periodo'], y=personal['valor'],
    mode='lines+markers',
    line=dict(color=C_SALUD, width=3),
    marker=dict(size=9, color=C_SALUD, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(76,159,56,0.08)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Personal sanitario: <b>%{y:.1f}</b> /10k hab.<extra></extra>'
), row=2, col=2)

fig_trab.add_annotation(
    x=2012, y=47.3, text='<b>47.3</b>\n2012',
    showarrow=False, font=dict(size=10, color=C_NEUTRO), row=2, col=2
)
fig_trab.add_annotation(
    x=2022, y=56.6, text='<b>56.6</b>\n2022',
    showarrow=False, font=dict(size=10, color=C_SALUD), row=2, col=2
)

fig_trab.add_annotation(
    x=2015, y=52.5,
    text='ODS 3-8: personal sanitario\ncorrelaciona inversamente\ncon empleo informal<br>(r = -0.98)',
    showarrow=False, font=dict(size=10, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ALERTA, borderwidth=1,
    row=2, col=2
)

fig_trab.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig_trab.update_yaxes(showgrid=True, gridcolor='#E9ECEF')

fig_trab.update_layout(
    title=dict(
        text='<b>ODS 8 — El Trabajo y sus Sombras</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             '1 de cada 2 trabajadores mexicanos esta en la informalidad — sin proteccion social ni ambiental</span>',
        x=0.5, xanchor='center'
    ),
    height=580, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig_trab.show()
print('Empleo informal (2024):', '{:.1f}%'.format(get_serie('8.3.1')['valor'].iloc[-1]))
print('Desocupacion minimo historico (2024):', '{:.2f}%'.format(get_serie('8.5.2')['valor'].iloc[-1]))

Empleo informal (2024): 50.7%
Desocupacion minimo historico (2024): 2.70%


---
## Parte V — El Fertilizante: la bomba de tiempo del agua

Un indicador que casi nadie menciona en la discusión pública:  
El uso de **fertilizantes fosfatados** en México pasó de **13.6 kg/ha** en 2005 a **398 kg/ha** en 2022.  
Un incremento de **+2,826%** en 17 años.

Los fertilizantes en exceso se filtran a los cuerpos de agua, provocando **eutrofización** — el envenenamiento de ríos y lagos por exceso de nutrientes que mata la biodiversidad acuática y hace el agua no apta para consumo humano.  

Esta es la conexión entre el modelo agrícola/laboral (ODS 8-12) y la crisis del agua (ODS 6).

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 6 — Fertilizantes + Cambio en cuerpos de agua
# ══════════════════════════════════════════════════════════════════════════════

fertil  = get_serie('12N.2.1')   # fertilizantes fosfatados
rio_per = get_serie('6.6.1.A')   # cambio area rios permanentes
rio_est = get_serie('6.6.1.B')   # cambio area rios estacionales
mangl   = get_serie('6.6.1.C')   # cambio area manglares

fig_fert = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Uso de Fertilizantes Fosfatados (kg/ha) — ODS 12n.2.1',
        'Cambio en Area de Cuerpos de Agua (%) — ODS 6.6.1'
    ],
    horizontal_spacing=0.12
)

# ── Panel izq: Fertilizantes ──────────────────────────────────────────────────
colors_fert = [
    C_ALERTA if v > 200 else '#E5891A' if v > 50 else C_MEJORA
    for v in fertil['valor']
]
fig_fert.add_trace(go.Bar(
    x=fertil['periodo'], y=fertil['valor'],
    marker_color=colors_fert,
    marker_line_color='white', marker_line_width=0.5,
    text=['{:.0f}'.format(v) for v in fertil['valor']],
    textposition='outside',
    textfont=dict(size=10),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Fertilizante: <b>%{y:.1f} kg/ha</b><extra></extra>'
), row=1, col=1)

fig_fert.add_annotation(
    x=2005, y=13.6, text='<b>13.6</b><br>2005',
    showarrow=True, arrowhead=2, arrowcolor=C_MEJORA,
    ax=50, ay=-30, font=dict(size=10, color=C_MEJORA),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_MEJORA, borderwidth=1,
    row=1, col=1
)
fig_fert.add_annotation(
    x=2022, y=398.0, text='<b>398 kg/ha</b><br>2022<br>+2,826%',
    showarrow=True, arrowhead=2, arrowcolor=C_ALERTA,
    ax=-60, ay=-40, font=dict(size=11, color=C_ALERTA),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ALERTA, borderwidth=1.5,
    row=1, col=1
)

# Nota: dato de 2022 es un salto (posible cambio metodologico o dato extremo)
fig_fert.add_annotation(
    x=2016, y=250,
    text='Nota: el dato de 2022 requiere<br>verificacion metodologica',
    showarrow=False, font=dict(size=9, color=C_NEUTRO),
    bgcolor='rgba(255,255,255,0.8)', bordercolor='#ADB5BD', borderwidth=1,
    row=1, col=1
)

# ── Panel der: Cambio en cuerpos de agua ──────────────────────────────────────
water_bodies = [
    (rio_per, 'Rios permanentes',  C_AGUA,   'circle'),
    (rio_est, 'Rios estacionales', C_TRABAJO,'square'),
    (mangl,   'Manglares',        C_SALUD,  'diamond'),
]
for data, name, color, sym in water_bodies:
    if len(data) > 0:
        fig_fert.add_trace(go.Scatter(
            x=data['periodo'], y=data['valor'],
            mode='lines+markers', name=name,
            line=dict(color=color, width=2.5),
            marker=dict(size=10, color=color, symbol=sym,
                        line=dict(width=1.5, color='white')),
            hovertemplate='<b>' + name + ' %{x}</b><br>Cambio: <b>%{y:+.1f}%</b><extra></extra>'
        ), row=1, col=2)

fig_fert.add_hline(
    y=0, line_width=1.5, line_color='#ADB5BD', row=1, col=2,
    annotation_text='Sin cambio', annotation_position='right',
    annotation_font=dict(size=9, color=C_NEUTRO)
)

fig_fert.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig_fert.update_yaxes(showgrid=True, gridcolor='#E9ECEF')
fig_fert.update_yaxes(title_text='Fertilizante (kg/ha)', row=1, col=1)
fig_fert.update_yaxes(title_text='Cambio porcentual en area (%)', row=1, col=2)

fig_fert.update_layout(
    title=dict(
        text='<b>La Bomba de Tiempo Agricola — Fertilizantes y Cuerpos de Agua</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'El uso de fertilizantes se dispara; los cuerpos de agua oscilan en inestabilidad</span>',
        x=0.5, xanchor='center'
    ),
    height=460, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(x=0.56, y=0.95, bgcolor='rgba(248,249,250,0.9)', borderwidth=1),
    margin=dict(t=100, b=60)
)
fig_fert.show()

---
## Parte VI — El Coste en Salud del Agua Insalubre

### ODS 3.9.2: Mortalidad por agua insalubre (WASH)

Cada año, personas en México mueren por enfermedades directamente atribuibles a agua insalubre, saneamiento deficiente y falta de higiene.  

La buena noticia: este indicador ha bajado consistentemente de 20.6 a 7.4 por 100,000 habitantes.  
La mala noticia: **el ritmo de descenso se está frenando** — y las correlaciones muestran que el consumo material creciente lo mantiene bajo presión permanente.

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 7 — Mortalidad WASH: descenso lento + comparativa con consumo
# ══════════════════════════════════════════════════════════════════════════════

wash  = get_serie('3.9.2A')    # mortalidad agua insalubre
cmat  = get_serie('12.2.2.B')  # consumo material

fig_wash = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Mortalidad por Agua Insalubre y Saneamiento Deficiente (por 100,000 hab.)',
        'Ritmo de Descenso — ¿se esta frenando?'
    ],
    horizontal_spacing=0.12
)

# ── Panel izq: Serie completa WASH ───────────────────────────────────────────
# Dividir en 3 fases para analizar el ritmo
wash_f1 = wash[wash['periodo'] <= 2008]  # descenso rapido
wash_f2 = wash[(wash['periodo'] > 2008) & (wash['periodo'] <= 2015)]  # descenso moderado
wash_f3 = wash[wash['periodo'] > 2015]   # descenso lento

for fase, nombre, color, dash in [
    (wash_f1, 'Descenso rapido (1998-2008)',   C_MEJORA, 'solid'),
    (wash_f2, 'Descenso moderado (2009-2015)', C_TRABAJO, 'solid'),
    (wash_f3, 'Descenso lento (2016-2023)',    C_ALERTA, 'solid'),
]:
    if len(fase) > 0:
        fig_wash.add_trace(go.Scatter(
            x=fase['periodo'], y=fase['valor'],
            mode='lines+markers', name=nombre,
            line=dict(color=color, width=3, dash=dash),
            marker=dict(size=7, color=color, line=dict(width=1.5, color='white')),
            hovertemplate='<b>%{x}</b><br>Mortalidad WASH: <b>%{y:.2f}</b>/100k<extra></extra>'
        ), row=1, col=1)

fig_wash.add_annotation(
    x=1998, y=20.6, text='<b>20.6</b><br>1998',
    showarrow=False, font=dict(size=10, color=C_NEUTRO), row=1, col=1
)
fig_wash.add_annotation(
    x=2023, y=7.38, text='<b>7.4</b><br>2023',
    showarrow=False, font=dict(size=10, color=C_ALERTA), row=1, col=1
)

# ── Panel der: tasa de cambio anual ──────────────────────────────────────────
wash_sorted = wash.sort_values('periodo').copy()
wash_sorted['cambio'] = wash_sorted['valor'].diff()
wash_sorted = wash_sorted.dropna()

colors_cambio = [C_MEJORA if v < 0 else C_ALERTA for v in wash_sorted['cambio']]
fig_wash.add_trace(go.Bar(
    x=wash_sorted['periodo'], y=wash_sorted['cambio'],
    marker_color=colors_cambio,
    marker_line_color='white', marker_line_width=0.5,
    name='Cambio anual',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Cambio: <b>%{y:+.2f}</b> /100k<extra></extra>'
), row=1, col=2)

fig_wash.add_hline(y=0, line_width=1.5, line_color='#ADB5BD', row=1, col=2)

# Tasa media por fase
for inicio, fin, y_pos, color, label in [
    (1999, 2008, 0.4, C_MEJORA,  'Media 1999-2008: -0.87/año'),
    (2009, 2015, 0.4, C_TRABAJO, 'Media 2009-2015: -0.29/año'),
    (2016, 2023, 0.4, C_ALERTA,  'Media 2016-2023: -0.23/año'),
]:
    f = wash_sorted[(wash_sorted['periodo'] >= inicio) & (wash_sorted['periodo'] <= fin)]
    if len(f) > 0:
        avg = f['cambio'].mean()
        fig_wash.add_annotation(
            x=(inicio + fin) // 2, y=y_pos,
            text='<b>' + '{:.2f}/año'.format(avg) + '</b>',
            showarrow=False, font=dict(size=10, color=color),
            bgcolor='rgba(255,255,255,0.85)', bordercolor=color, borderwidth=1,
            row=1, col=2
        )

fig_wash.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig_wash.update_yaxes(showgrid=True, gridcolor='#E9ECEF')
fig_wash.update_yaxes(title_text='Muertes por 100,000 hab.', row=1, col=1)
fig_wash.update_yaxes(title_text='Cambio anual (por 100,000 hab.)', row=1, col=2)

fig_wash.update_layout(
    title=dict(
        text='<b>El Precio en Vidas del Agua Insalubre (ODS 3.9.2)</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Mortalidad por agua insalubre: mejora lenta que se esta desacelerando</span>',
        x=0.5, xanchor='center'
    ),
    height=460, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(x=0.01, y=0.02, bgcolor='rgba(248,249,250,0.9)', borderwidth=1),
    margin=dict(t=100, b=60)
)
fig_wash.show()

---
## Parte VII — El Mapa del Trilema

Síntesis visual: los tres ODS conectados por correlaciones negativas forman un **sistema de tensiones**.  
El diagrama de red cuantifica cuántos indicadores de cada par están negativamente correlacionados — y con qué fuerza.

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 8 — Mapa del trilema: heatmap de correlaciones cruzadas
# ══════════════════════════════════════════════════════════════════════════════

import re

def ods_num(ind_id):
    clean = re.sub(r'[NRnr]', '', str(ind_id))
    m = re.match(r'^(\d+)', clean)
    return int(m.group(1)) if m else None

corr['ods_1n'] = corr['ind_1'].apply(ods_num)
corr['ods_2n'] = corr['ind_2'].apply(ods_num)

# ── Panel izq: Heatmap de correlaciones promedio entre ODS trilema ────────────
ods_trilema = [3, 6, 8, 12]
ods_labels  = ['ODS 3\nSalud', 'ODS 6\nAgua', 'ODS 8\nTrabajo', 'ODS 12\nConsumo']

matrix_mean = np.zeros((4, 4))
matrix_count = np.zeros((4, 4))
matrix_text  = [[''] * 4 for _ in range(4)]

for i, ods_a in enumerate(ods_trilema):
    for j, ods_b in enumerate(ods_trilema):
        if i == j:
            matrix_mean[i][j] = 0
            matrix_text[i][j] = '—'
            continue
        pairs = corr[
            ((corr['ods_1n']==ods_a) & (corr['ods_2n']==ods_b)) |
            ((corr['ods_1n']==ods_b) & (corr['ods_2n']==ods_a))
        ]
        if len(pairs) > 0:
            avg = pairs['correlation'].mean()
            matrix_mean[i][j] = avg
            matrix_count[i][j] = len(pairs)
            matrix_text[i][j] = '{:.2f}<br>(n={})'.format(avg, len(pairs))
        else:
            matrix_text[i][j] = 'sin datos'

fig_trilema = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Correlacion Spearman Promedio entre ODS del Trilema',
        'Numero de Pares con Correlacion Fuerte (|r|>=0.7)'
    ],
    horizontal_spacing=0.12
)

# Heatmap de correlaciones promedio
fig_trilema.add_trace(go.Heatmap(
    z=matrix_mean,
    x=ods_labels, y=ods_labels,
    text=matrix_text,
    texttemplate='%{text}',
    textfont=dict(size=11),
    colorscale='RdYlGn',
    zmid=0, zmin=-0.5, zmax=0.5,
    showscale=True,
    colorbar=dict(title='r Spearman', x=0.46, thickness=15),
    hovertemplate='<b>%{y} vs %{x}</b><br>Correlacion media: <b>%{z:.3f}</b><extra></extra>'
), row=1, col=1)

# Heatmap de número de pares
matrix_count_display = matrix_count.copy()
np.fill_diagonal(matrix_count_display, np.nan)
count_text = [[str(int(v)) if not np.isnan(v) else '—'
               for v in row] for row in matrix_count_display]

fig_trilema.add_trace(go.Heatmap(
    z=matrix_count_display,
    x=ods_labels, y=ods_labels,
    text=count_text,
    texttemplate='%{text}',
    textfont=dict(size=13, color='white'),
    colorscale='Blues',
    showscale=True,
    colorbar=dict(title='N pares', x=1.0, thickness=15),
    hovertemplate='<b>%{y} vs %{x}</b><br>Pares fuertes: <b>%{z:.0f}</b><extra></extra>'
), row=1, col=2)

fig_trilema.update_layout(
    title=dict(
        text='<b>El Mapa del Trilema — ODS 3 / 6 / 8 / 12</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Rojo = correlacion negativa (mas crecimiento economico = peores indicadores de agua/salud)</span>',
        x=0.5, xanchor='center'
    ),
    height=440, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig_trilema.show()

print('Resumen del trilema:')
for i, ods_a in enumerate(ods_trilema):
    for j, ods_b in enumerate(ods_trilema):
        if j > i:
            avg = matrix_mean[i][j]
            n   = int(matrix_count[i][j])
            dir_s = 'NEGATIVA' if avg < -0.05 else 'POSITIVA' if avg > 0.05 else 'NEUTRAL'
            print('  ODS {} vs ODS {}: {} (r={:.3f}, n={})'.format(ods_a, ods_b, dir_s, avg, n))

Resumen del trilema:
  ODS 3 vs ODS 6: NEUTRAL (r=-0.005, n=97)
  ODS 3 vs ODS 8: NEGATIVA (r=-0.068, n=177)
  ODS 3 vs ODS 12: NEGATIVA (r=-0.273, n=48)
  ODS 6 vs ODS 8: NEGATIVA (r=-0.075, n=88)
  ODS 6 vs ODS 12: NEUTRAL (r=-0.043, n=29)
  ODS 8 vs ODS 12: POSITIVA (r=0.326, n=44)


---
## Sintesis Final — El Ciclo Vicioso

El **diagrama de flujo del trilema** muestra cómo los tres ODS se retroalimentan negativamente:

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 9 — Diagrama conceptual del ciclo vicioso (Sankey adaptado)
# ══════════════════════════════════════════════════════════════════════════════

fig_sankey = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        pad=20, thickness=28,
        line=dict(color='white', width=0.5),
        label=[
            'ODS 8: Crecimiento Economico',    # 0
            'ODS 12: Consumo Material x18',    # 1
            'ODS 8: Empleo Informal 51%',      # 2
            'ODS 12: Fertilizantes +2826%',    # 3
            'ODS 6: Estres Hidrico 47%',       # 4
            'ODS 6: Agua Segura -6 pp',        # 5
            'ODS 6: Residuales sin tratar 60%',# 6
            'ODS 3: Mort. WASH 7.4/100k',      # 7
            'ODS 3: Enf. Cronicas +22%',       # 8
            'ODS 3: Sin Seguridad Social',     # 9
        ],
        color=[
            C_TRABAJO, C_CONSUMO, C_TRABAJO, C_CONSUMO,
            C_AGUA, C_AGUA, C_AGUA,
            C_ALERTA, C_ALERTA, C_ALERTA
        ],
        x=[0.05, 0.05, 0.05, 0.05,
           0.42, 0.42, 0.42,
           0.78, 0.78, 0.78],
        y=[0.15, 0.40, 0.65, 0.85,
           0.20, 0.50, 0.78,
           0.15, 0.50, 0.82]
    ),
    link=dict(
        source=[0, 0, 1, 2, 3,  4, 5, 6, 4],
        target=[1, 2, 4, 9, 4,  5, 7, 7, 6],
        value= [8, 5, 7, 5, 6,  5, 4, 4, 4],
        color=[
            'rgba(229,137,26,0.35)',  # 0->1
            'rgba(229,137,26,0.25)',  # 0->2
            'rgba(139,26,26,0.30)',   # 1->4
            'rgba(229,137,26,0.20)',  # 2->9
            'rgba(139,26,26,0.25)',   # 3->4
            'rgba(29,111,164,0.30)',  # 4->5
            'rgba(76,159,56,0.25)',   # 5->7
            'rgba(29,111,164,0.25)',  # 6->7
            'rgba(29,111,164,0.20)',  # 4->6
        ],
        label=[
            'impulsa', 'genera', 'presiona (r=-1.0)', 'excluye',
            'contamina (r=-0.92)', 'reduce', 'causa muertes',
            'vierte sin tratar', 'estresa'
        ],
        hovertemplate='<b>%{source.label}</b><br>→ %{target.label}<br>'
                      '<i>%{label}</i><extra></extra>'
    )
))

fig_sankey.update_layout(
    title=dict(
        text='<b>El Ciclo Vicioso del Trilema Trabajo-Agua-Salud</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Flujo causal basado en correlaciones ODS de Mexico (datos INEGI 2026)</span>',
        x=0.5, xanchor='center'
    ),
    height=500, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA, size=12),
    margin=dict(t=100, b=60, l=20, r=20)
)
fig_sankey.show()

---
## Conclusiones — El Diagnóstico del Trilema

### Lo que los datos revelan sobre el modelo de desarrollo de México

---

**1. El crecimiento económico de México tiene un costo hídrico no contabilizado.**  
El consumo material x18 en 20 años correlaciona perfectamente (r = −1.0) con el retroceso en acceso a agua potable segura. Esto no es coincidencia — es la huella hídrica de un modelo productivo que externaliza su costo ambiental.

**2. México ya superó el umbral crítico de estrés hídrico.**  
El 47% de extracción sobre disponibilidad (40% es el límite de alerta) significa que el país opera sin margen de seguridad hídrica. Las zonas Centro y Norte están en 56% — territorio de escasez crónica.

**3. El retroceso en tratamiento de aguas residuales es una regresión ambiental grave.**  
Caer del 50% al 40% en 5 años significa que 60 de cada 100 litros de aguas residuales regresan a los cuerpos de agua sin tratamiento. Este contaminante es el vínculo directo con la mortalidad WASH.

**4. El empleo informal es el puente invisible entre trabajo y salud hídrica.**  
Con el 51% de trabajadores en la informalidad, millones de personas no tienen acceso a seguridad social, agua potable regulada en el trabajo ni protección frente a exposición a contaminantes hídricos.

**5. El uso de fertilizantes es la amenaza hídrica menos discutida.**  
Un aumento de +2,826% en 17 años (con verificación pendiente del dato 2022) representa una presión de eutrofización sobre los cuerpos de agua que los indicadores actuales aún no capturan completamente.

---

### ¿Qué necesita México para romper el ciclo?

| Prioridad | Acción concreta |
|-----------|----------------|
| **Inmediata** | Revertir el retroceso en tratamiento de aguas residuales: inversión urgente en infraestructura |
| **Urgente** | Reducir el estrés hídrico en zonas C/N con tarifas diferenciadas y tecnología de eficiencia |
| **Estructural** | Formalizar el empleo: extender seguridad social a los 15 millones de trabajadores informales |
| **Regulatoria** | Controlar el uso de fertilizantes con límites por cuenca hidrológica |
| **Sistemica** | Incorporar el costo del agua en los precios de producción — hoy es una externalidad invisible |

---

**El mensaje central:** México no puede seguir midiendo su progreso solo con PIB y empleo.  
Un crecimiento que destruye el agua no es desarrollo — es deuda ambiental diferida.

---

*Análisis basado en indicadores ODS 3, 6, 8 y 12 de México · INEGI 2026 · HackODS 2026*  
*Método: correlaciones Spearman, análisis de tendencias, diagramas de flujo causal*